# HoTHP Main Result: Horizon Extrapolation

**Run on Colab: Runtime → Change runtime type → T4 GPU**

---

**The discovery in one sentence:**  
HoTHP's monotonic attention kernel guarantees stable OOD performance when trained on short sequences and deployed on much longer ones — but only when the underlying process has fast-decaying influence. For slow-decay processes, neither model has an advantage.

**Process parameters chosen from prior sweep results:**
- Slow decay: β = 0.02  →  β_norm ≈ 0.025  →  influence at 10× horizon ≈ 0.29
- Fast decay: β = 0.50  →  β_norm ≈ 0.40   →  influence at 10× horizon ≈ 0.00

**Expected runtime:** ~25–35 min on T4 GPU  
(2 processes × 5 seeds × 2 models × 1 restart)

In [ ]:
import os
if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git
!pip install omegaconf -q

In [ ]:
import os, sys, math, random, hashlib, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from scipy import stats

# ── Parameters ────────────────────────────────────────────────────────────
TRAIN_LEN      = 50          # max events per training sequence
EXTRAP_FACTORS = [2, 5, 10]  # test horizons: 100 / 250 / 500 events
N_SEEDS        = 5
N_RESTARTS     = 1
EPOCHS         = 500
PATIENCE       = 30
BASE_SEED      = 42
NUM_TYPES      = 2
PAD_ID         = NUM_TYPES

# Process definitions (chosen from sweep: confirmed null / confirmed significant)
PROC_SLOW = dict(
    mu=np.array([0.3, 0.3]),
    alpha=np.array([[0.008, 0.006], [0.006, 0.008]]),
    beta=0.02,
    label='Slow decay  (β_norm ≈ 0.025)',
    color_rothp='#4C72B0',
    color_hothp='#C44E52',
)
PROC_FAST = dict(
    mu=np.array([0.4, 0.4]),
    alpha=np.array([[0.12, 0.08], [0.08, 0.12]]),
    beta=0.50,
    label='Fast decay  (β_norm ≈ 0.40)',
    color_rothp='#4C72B0',
    color_hothp='#C44E52',
)
PROCESSES = [('slow', PROC_SLOW), ('fast', PROC_FAST)]
# ─────────────────────────────────────────────────────────────────────────

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_seed(*parts):
    key = '::'.join(map(str, parts))
    return (BASE_SEED + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_seed(BASE_SEED)
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def _attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dim() == 3: mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p = torch.softmax(scores, dim=-1)
    if dropout is not None: p = dropout(p)
    return torch.matmul(p, value), p

baselayer.attention = _attention
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = _attention

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

USE_AMP = device.type == 'cuda'
if USE_AMP:
    try:
        _autocast = lambda: torch.amp.autocast(device_type='cuda')
        _Scaler   = torch.amp.GradScaler
    except AttributeError:
        _autocast = torch.cuda.amp.autocast
        _Scaler   = torch.cuda.amp.GradScaler
else:
    _autocast = contextlib.nullcontext
    _Scaler   = None

print(f'Device: {device}  |  AMP: {USE_AMP}')
print(f'Train max: {TRAIN_LEN} events  →  OOD zone starts at position {TRAIN_LEN}')
print(f'Extrap horizons: {[TRAIN_LEN*f for f in EXTRAP_FACTORS]} events ({EXTRAP_FACTORS}×)')

In [ ]:
# ── Data generation ───────────────────────────────────────────────────────

def simulate_hawkes(rng, mu, alpha, beta, horizon, min_ev, max_ev):
    for _ in range(100):
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9: break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon: break
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
            if rng.uniform() <= cand.sum() / lam_bar:
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


def to_tensors(seqs):
    """Per-sequence prefix normalisation: mean inter-event gap → 1."""
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)
        d = torch.zeros_like(t)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)
        t = (t - t[0]) / mg
        d = d / mg
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


def collate(batch):
    B = len(batch)
    L = max(len(x['time_seqs']) for x in batch)
    t_pad  = torch.zeros(B, L)
    d_pad  = torch.zeros(B, L)
    k_pad  = torch.full((B, L), PAD_ID, dtype=torch.long)
    npm    = torch.zeros(B, L)
    causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    attn   = torch.ones(B, L, L, dtype=torch.bool)
    for i, item in enumerate(batch):
        sl = len(item['time_seqs'])
        t_pad[i, :sl] = item['time_seqs']
        d_pad[i, :sl] = item['time_delta_seqs']
        k_pad[i, :sl] = item['type_seqs']
        npm[i, :sl]   = 1.0
        m = causal.clone(); m[:, sl:] = True; m[sl:, :] = True
        attn[i] = m
    return t_pad, d_pad, k_pad, npm, attn


def make_loader(data, bs, shuffle=False, seed=None):
    g = None
    if shuffle and seed is not None:
        g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(data, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, generator=g)


print('Generating datasets...')
rng = np.random.default_rng(run_seed('main', 'data'))
datasets = {}

for pname, proc in PROCESSES:
    mu, alpha, beta = proc['mu'], proc['alpha'], proc['beta']

    raw = {
        'train': [simulate_hawkes(rng, mu, alpha, beta, 50.0,      10, TRAIN_LEN)      for _ in range(500)],
        'val':   [simulate_hawkes(rng, mu, alpha, beta, 50.0,      10, TRAIN_LEN)      for _ in range(150)],
        'short': [simulate_hawkes(rng, mu, alpha, beta, 50.0,      10, TRAIN_LEN)      for _ in range(200)],
    }
    for f in EXTRAP_FACTORS:
        raw[f'extrap_{f}x'] = [
            simulate_hawkes(rng, mu, alpha, beta,
                            50.0 * f, TRAIN_LEN + 5, TRAIN_LEN * f)
            for _ in range(200)
        ]

    datasets[pname] = {k: to_tensors(v) for k, v in raw.items()}

    # Measure actual β_norm
    gaps = []
    for seq in raw['train']:
        ts = sorted([t for t, _ in seq])
        gaps.extend([ts[i] - ts[i-1] for i in range(1, len(ts))])
    mg   = np.mean(gaps)
    bn   = beta * mg
    infl = math.exp(-bn * (TRAIN_LEN * max(EXTRAP_FACTORS) - 1))
    print(f'  {pname:4s}  beta={beta:.2f}  mean_gap={mg:.3f}  '
          f'β_norm={bn:.4f}  influence@10x={infl:.4f}')

print('\nOOD fraction (events at positions ≥ TRAIN_LEN):')
for f in EXTRAP_FACTORS:
    seqs  = datasets['slow'][f'extrap_{f}x']
    total = sum(len(s['time_seqs']) for s in seqs)
    ood   = sum(max(0, len(s['time_seqs']) - TRAIN_LEN) for s in seqs)
    print(f'  {f}×: {100*ood/total:.0f}% of events are OOD')

In [ ]:
# ── Model config and training ─────────────────────────────────────────────

config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': PAD_ID, 'time_emb_size': 32, 'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'MainResult',
    'thinning': {'num_sample':1,'num_exp':500,'over_sample_rate':5.0,
                 'patience_counter':5,'num_samples_boundary':5,'dtime_max':5.0,'num_step_gen':1},
    'loss_integral_num_sample_per_step': 20, 'use_mc_samples': False,
})


def eval_nll(model, dl):
    model.eval(); total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            with _autocast():
                l, n = model.loglike_loss(batch)
            total_l += l.item(); total_n += n
    return total_l / (total_n + 1e-9)


def eval_ood_nll(model, dl):
    """NLL only for events at positions >= TRAIN_LEN."""
    model.eval(); total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            t, d, k, npm, attn = [x.to(device) for x in batch]
            mask = npm.clone()
            mask[:, :TRAIN_LEN] = 0.0
            if mask.sum() == 0: continue
            with _autocast():
                l, n = model.loglike_loss([t, d, k, mask, attn])
            total_l += l.item(); total_n += n
    return total_l / (total_n + 1e-9)


def train_model(cls, train_dl, val_dl, lr, base_seed):
    best_val, best_state = float('inf'), None
    for r in range(N_RESTARTS):
        set_seed(base_seed + r * 7919)
        m = cls(config).to(device)
        opt   = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', factor=0.5, patience=10, min_lr=1e-5)
        scaler  = _Scaler(enabled=True) if USE_AMP else None
        no_imp  = 0
        best_v  = float('inf')
        best_st = None
        for ep in range(EPOCHS):
            m.train()
            for batch in train_dl:
                batch = [t.to(device) for t in batch]
                opt.zero_grad()
                with _autocast():
                    l, n = m.loglike_loss(batch)
                    nll  = l / (n + 1e-9)
                if not torch.isnan(nll):
                    if scaler:
                        scaler.scale(nll).backward()
                        scaler.unscale_(opt)
                        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                        scaler.step(opt); scaler.update()
                    else:
                        nll.backward()
                        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                        opt.step()
            v = eval_nll(m, val_dl)
            sched.step(v)
            if v < best_v - 1e-4:
                best_v = v
                best_st = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
                no_imp = 0
            else:
                no_imp += 1
            if no_imp >= PATIENCE: break
        if best_v < best_val:
            best_val   = best_v
            best_state = best_st
    m = cls(config).to(device)
    m.load_state_dict(best_state)
    return m, best_val


# ── Run experiment ────────────────────────────────────────────────────────
seeds   = [BASE_SEED + i * 100 for i in range(N_SEEDS)]
results = []

for pname, proc in PROCESSES:
    print(f'\n{"+"*55}')
    print(f'  {proc["label"]}')
    print(f'{"+"*55}')

    val_dl   = make_loader(datasets[pname]['val'],   64)
    short_dl = make_loader(datasets[pname]['short'], 64)
    ext_dls  = {f: make_loader(datasets[pname][f'extrap_{f}x'], 16)
                for f in EXTRAP_FACTORS}

    for seed_idx, seed in enumerate(seeds):
        print(f'  Seed {seed_idx+1}/{N_SEEDS}', end='  ')
        train_dl = make_loader(datasets[pname]['train'], 64,
                               shuffle=True, seed=run_seed(pname, seed))

        rothp, rv = train_model(RoTHP, train_dl, val_dl, lr=1e-3,
                                 base_seed=run_seed(pname, 'rothp', seed))
        hothp, hv = train_model(HoTHP, train_dl, val_dl, lr=5e-4,
                                 base_seed=run_seed(pname, 'hothp', seed))

        r_short = eval_nll(rothp, short_dl)
        h_short = eval_nll(hothp, short_dl)
        print(f'val: RoTHP={rv:.4f}  HoTHP={hv:.4f}')

        for f in EXTRAP_FACTORS:
            r_ood = eval_ood_nll(rothp, ext_dls[f])
            h_ood = eval_ood_nll(hothp, ext_dls[f])
            results.append(dict(
                proc=pname, seed=seed, factor=f,
                r_short=r_short, h_short=h_short,
                r_ood=r_ood,     h_ood=h_ood,
                r_deg=r_ood - r_short,
                h_deg=h_ood - h_short,
                adv=r_ood - h_ood,
            ))

df = pd.DataFrame(results)
print('\nDone.')

In [ ]:
# gráfico principal: quanto cada modelo degrada fora da distribuição de treino
# ΔNLL = NLL no teste OOD - NLL no treino
# se ΔNLL ≈ 0, o modelo não piorou ao extrapolar

C_ROTHP = '#4C72B0'
C_HOTHP = '#C44E52'
factor_labels = [f'{f}×' for f in EXTRAP_FACTORS]
x = np.arange(len(EXTRAP_FACTORS))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for col, (pname, proc) in enumerate(PROCESSES):
    sub = df[df['proc'] == pname]
    ax  = axes[col]

    for col_key, label, color in [
        ('r_deg', 'RoTHP', C_ROTHP),
        ('h_deg', 'HoTHP', C_HOTHP),
    ]:
        grp = sub.groupby('factor')[col_key]
        m   = grp.mean().reindex(EXTRAP_FACTORS)
        s   = grp.std().reindex(EXTRAP_FACTORS)
        ax.plot(x, m.values, 'o-', color=color, lw=2, ms=7, label=label)
        ax.fill_between(x, m - s, m + s, color=color, alpha=0.15)

    # y=0 significa que o modelo no teste OOD tem o mesmo NLL que no treino
    ax.axhline(0, color='gray', ls='--', lw=1.2, label='sem piora')

    ax.set_xticks(x)
    ax.set_xticklabels(factor_labels)
    ax.set_xlabel('fator de extrapolação')
    ax.set_ylabel('ΔNLL  (OOD − treino)')
    ax.set_title(proc['label'])
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f'HoTHP vs RoTHP — degradação com extrapolação\n'
    f'treino: ≤{TRAIN_LEN} eventos  →  teste: até {TRAIN_LEN * max(EXTRAP_FACTORS)} eventos  |  n={N_SEEDS} seeds'
)
plt.tight_layout()
plt.savefig('HoTHP_MainResult.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Numerical summary ─────────────────────────────────────────────────────

print('=' * 65)
print('NUMERICAL SUMMARY')
print('=' * 65)

for pname, proc in PROCESSES:
    sub = df[df['proc'] == pname]
    print(f'\n{proc["label"]}')
    print(f'  {"Factor":>6}  {"RoTHP OOD":>11}  {"HoTHP OOD":>11}  '
          f'{"Advantage":>11}  {"p":>7}  sig')
    for f in EXTRAP_FACTORS:
        row = sub[sub['factor'] == f]
        rm  = row['r_ood'].mean(); rs = row['r_ood'].std()
        hm  = row['h_ood'].mean(); hs = row['h_ood'].std()
        adv = row['adv'].values
        t_stat, p2 = stats.ttest_1samp(adv, 0)
        p1  = p2 / 2 if t_stat > 0 else 1 - p2 / 2
        sig = '***' if p1 < 0.001 else ('**' if p1 < 0.01 else
              ('*' if p1 < 0.05 else ('~' if p1 < 0.10 else 'ns')))
        print(f'  {f}×     {rm:.4f}±{rs:.3f}  {hm:.4f}±{hs:.3f}  '
              f'{adv.mean():>+.4f}±{adv.std():.3f}  p={p1:.3f}  {sig}')

In [ ]:
# ── Presentation figure (fancy) ───────────────────────────────────────────
# Self-contained. Uses: df, EXTRAP_FACTORS, TRAIN_LEN, N_SEEDS, PROCESSES
# Saves: HoTHP_Presentation.png
# ─────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy import stats as _stats

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        13,
    "axes.titlesize":   15,
    "axes.labelsize":   13,
    "xtick.labelsize":  12,
    "ytick.labelsize":  12,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.color":       "#e8e8e8",
    "grid.linewidth":   0.8,
})

C_RO   = "#1565C0"   # blue  — RoTHP
C_HO   = "#B71C1C"   # red   — HoTHP
OUTLINE = [pe.withStroke(linewidth=4, foreground="white")]
XPOS    = np.arange(len(EXTRAP_FACTORS))

def sig_stars(adv_vals):
    t, p2 = _stats.ttest_1samp(adv_vals, 0)
    p1 = p2 / 2 if t > 0 else 1.0
    if p1 < 0.001: return "***"
    if p1 < 0.01:  return "**"
    if p1 < 0.05:  return "*"
    return "ns"

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.subplots_adjust(wspace=0.38)

PROC_TITLES = {
    "slow": "Slow-decay  (β_norm ≈ 0.025)",
    "fast": "Fast-decay  (β_norm ≈ 0.40)",
}

for ax, (pname, proc) in zip(axes, PROCESSES):
    sub      = df[df["proc"] == pname]
    is_fast  = (pname == "fast")

    ro_means, ho_means = [], []
    ro_lows,  ho_lows  = [], []
    ro_highs, ho_highs = [], []

    for fi, f in enumerate(EXTRAP_FACTORS):
        rows     = sub[sub["factor"] == f]
        ro_delta = rows["r_deg"].values        # r_ood - r_short (ΔNLL)
        ho_delta = rows["h_deg"].values

        ro_means.append(ro_delta.mean())
        ho_means.append(ho_delta.mean())

        for vals, lows, highs in [(ro_delta, ro_lows, ro_highs),
                                   (ho_delta, ho_lows, ho_highs)]:
            ci = _stats.t.interval(0.95, len(vals) - 1,
                                   loc=vals.mean(),
                                   scale=_stats.sem(vals))
            lows.append(ci[0]); highs.append(ci[1])

        # per-seed scatter (subtle)
        jit = np.random.uniform(-0.10, 0.10, len(ro_delta))
        ax.scatter(fi + jit, ro_delta, color=C_RO, alpha=0.20, s=25, zorder=2)
        ax.scatter(fi + jit, ho_delta, color=C_HO, alpha=0.20, s=25, zorder=2)

    ro_means = np.array(ro_means)
    ho_means = np.array(ho_means)

    # Lines with white outline for visual pop
    ax.plot(XPOS, ro_means, "-o", color=C_RO, lw=2.8, ms=9,
            label="RoTHP", zorder=5, path_effects=OUTLINE)
    ax.plot(XPOS, ho_means, "-o", color=C_HO, lw=2.8, ms=9,
            label="HoTHP", zorder=5, path_effects=OUTLINE)

    ax.fill_between(XPOS, ro_lows, ro_highs, color=C_RO, alpha=0.12)
    ax.fill_between(XPOS, ho_lows, ho_highs, color=C_HO, alpha=0.12)

    # Green shaded advantage zone (fast panel only, where RoTHP > HoTHP)
    if is_fast:
        ax.fill_between(XPOS, ho_means, ro_means,
                        where=(ro_means > ho_means),
                        color="#2E7D32", alpha=0.14,
                        label="HoTHP advantage", interpolate=True)

    # Significance stars above the higher line at each factor
    y_range = ax.get_ylim()[1] - ax.get_ylim()[0] if ax.get_ylim()[1] != ax.get_ylim()[0] else 1
    for fi, f in enumerate(EXTRAP_FACTORS):
        rows  = sub[sub["factor"] == f]
        stars = sig_stars(rows["adv"].values)
        if stars == "ns":
            continue
        top_y = max(ro_means[fi], ho_means[fi])
        ax.text(fi, top_y + 0.04, stars,
                ha="center", va="bottom", fontsize=13,
                fontweight="bold", color="#333")

    # Callout box for fast panel at 10× with the key number
    if is_fast:
        last_f = EXTRAP_FACTORS[-1]
        rows   = sub[sub["factor"] == last_f]
        adv_mu = rows["adv"].values.mean()
        t_, p2 = _stats.ttest_1samp(rows["adv"].values, 0)
        p1     = p2 / 2 if t_ > 0 else 1.0
        label  = f"+{adv_mu:.3f} nats\n@ {last_f}× horizon\np < 0.001" if p1 < 0.001 \
                 else f"+{adv_mu:.3f} nats\n@ {last_f}× horizon\np = {p1:.3f}"
        ax.annotate(
            label,
            xy=(XPOS[-1], ro_means[-1]),
            xytext=(-70, 28), textcoords="offset points",
            fontsize=11, color="#1a237e",
            bbox=dict(boxstyle="round,pad=0.5", fc="#E3F2FD",
                      ec="#1565C0", lw=1.5),
            arrowprops=dict(arrowstyle="->", color="#1565C0", lw=1.5),
        )

    ax.axhline(0, color="#888", lw=1.2, ls="--", zorder=1, label="no degradation")
    ax.set_xticks(XPOS)
    ax.set_xticklabels([f"{f}×" for f in EXTRAP_FACTORS])
    ax.set_xlabel("Extrapolation factor  (test / train horizon)", labelpad=8)
    ax.set_ylabel("ΔNLL  =  OOD NLL − in-dist NLL  (nats)", labelpad=8)
    ax.set_title(PROC_TITLES[pname], fontweight="bold", pad=12)
    ax.legend(loc="upper left", framealpha=0.9, fontsize=11)

fig.suptitle(
    "HoTHP vs RoTHP — Horizon Extrapolation Reliability\n"
    f"Train ≤ {TRAIN_LEN} events  →  test up to {TRAIN_LEN * max(EXTRAP_FACTORS)} events"
    f"  |  n = {N_SEEDS} seeds  |  ΔNLL = 0 means no degradation",
    fontsize=13, fontweight="bold", y=1.03,
)

plt.tight_layout()
plt.savefig("HoTHP_Presentation.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved → HoTHP_Presentation.png")

In [ ]:
# ── Training Efficiency: "Train Short, Test Long" ─────────────────────────
#
# Analogous to ALiBi (Press et al., 2022):
#   "A model trained on L events and extrapolated to L*f events achieves
#    similar NLL to a model trained directly on L*f events,
#    at a fraction of the compute."
#
# Protocol (fast-decay process only, where HoTHP's advantage is established):
#   SHORT model: trained on ≤ TRAIN_LEN events  (already done — reuse df)
#   ORACLE model: trained on ≤ TRAIN_LEN * ORACLE_FACTOR events
#   Compare their OOD NLL on the 10× test set.
#   Report: wall-clock time ratio  +  NLL gap vs oracle.
#
# N_SEEDS_EFF seeds to keep runtime manageable (~10-15 min on T4).
# ─────────────────────────────────────────────────────────────────────────
import time

ORACLE_FACTOR  = 10          # oracle trains on sequences up to 10× TRAIN_LEN
ORACLE_LEN     = TRAIN_LEN * ORACLE_FACTOR   # 500 events
N_SEEDS_EFF    = 3
PNAME_EFF      = 'fast'
PROC_EFF       = PROC_FAST

print(f'Efficiency experiment — {PROC_EFF["label"]}')
print(f'  Short training : sequences ≤ {TRAIN_LEN} events')
print(f'  Oracle training: sequences ≤ {ORACLE_LEN} events')
print(f'  Evaluation     : OOD zone positions ≥ {TRAIN_LEN} on {ORACLE_FACTOR}× test set')
print(f'  n = {N_SEEDS_EFF} seeds\n')

# ── Generate long training sequences (oracle) ─────────────────────────────
rng_eff = np.random.default_rng(run_seed('efficiency', 'data'))
mu, alpha, beta = PROC_EFF['mu'], PROC_EFF['alpha'], PROC_EFF['beta']

oracle_train_raw = [
    simulate_hawkes(rng_eff, mu, alpha, beta,
                    horizon=200.0, min_ev=20, max_ev=ORACLE_LEN)
    for _ in range(500)
]
oracle_val_raw = [
    simulate_hawkes(rng_eff, mu, alpha, beta,
                    horizon=200.0, min_ev=20, max_ev=ORACLE_LEN)
    for _ in range(150)
]
oracle_train_data = to_tensors(oracle_train_raw)
oracle_val_data   = to_tensors(oracle_val_raw)

# Re-use the existing 10× test set from main experiment
test_10x_dl = make_loader(datasets[PNAME_EFF]['extrap_10x'], 16)

print(f'Oracle train set: {len(oracle_train_data)} seqs, '
      f'mean length = {np.mean([len(s["time_seqs"]) for s in oracle_train_data]):.1f} events')

# ── Helper: train with timing ─────────────────────────────────────────────
def train_timed(cls, train_data, val_data, lr, base_seed, bs=64):
    val_dl   = make_loader(val_data,   bs)
    train_dl = make_loader(train_data, bs, shuffle=True,
                            seed=run_seed('eff', base_seed))
    t0 = time.time()
    model, best_val = train_model(cls, train_dl, val_dl, lr=lr,
                                   base_seed=run_seed('eff', base_seed))
    elapsed = time.time() - t0
    return model, best_val, elapsed

# ── Run seeds ─────────────────────────────────────────────────────────────
eff_results = []
seeds_eff   = [BASE_SEED + i * 317 for i in range(N_SEEDS_EFF)]

for si, seed in enumerate(seeds_eff):
    print(f'Seed {si+1}/{N_SEEDS_EFF}')

    # SHORT models (already trained — but retrain here for timing)
    ro_short, _, t_ro_short = train_timed(RoTHP,
        datasets[PNAME_EFF]['train'], datasets[PNAME_EFF]['val'],
        lr=1e-3, base_seed=run_seed(PNAME_EFF, 'rothp', seed))
    ho_short, _, t_ho_short = train_timed(HoTHP,
        datasets[PNAME_EFF]['train'], datasets[PNAME_EFF]['val'],
        lr=5e-4, base_seed=run_seed(PNAME_EFF, 'hothp', seed))

    # ORACLE models (trained on long sequences)
    ro_oracle, _, t_ro_oracle = train_timed(RoTHP,
        oracle_train_data, oracle_val_data,
        lr=1e-3, base_seed=run_seed('oracle', 'rothp', seed), bs=16)
    ho_oracle, _, t_ho_oracle = train_timed(HoTHP,
        oracle_train_data, oracle_val_data,
        lr=5e-4, base_seed=run_seed('oracle', 'hothp', seed), bs=16)

    # Evaluate OOD NLL on 10× test set
    def _ood(m): return eval_ood_nll(m, test_10x_dl)

    eff_results.append(dict(
        seed          = seed,
        ro_short_ood  = _ood(ro_short),
        ho_short_ood  = _ood(ho_short),
        ro_oracle_ood = _ood(ro_oracle),
        ho_oracle_ood = _ood(ho_oracle),
        t_ro_short    = t_ro_short,
        t_ho_short    = t_ho_short,
        t_ro_oracle   = t_ro_oracle,
        t_ho_oracle   = t_ho_oracle,
    ))
    print(f'  RoTHP  short={eff_results[-1]["ro_short_ood"]:.4f}  '
          f'oracle={eff_results[-1]["ro_oracle_ood"]:.4f}  '
          f'time ratio={t_ro_oracle/t_ro_short:.1f}×')
    print(f'  HoTHP  short={eff_results[-1]["ho_short_ood"]:.4f}  '
          f'oracle={eff_results[-1]["ho_oracle_ood"]:.4f}  '
          f'time ratio={t_ho_oracle/t_ho_short:.1f}×')

edf = pd.DataFrame(eff_results)

# ── Summary ───────────────────────────────────────────────────────────────
print('\n' + '='*70)
print(f'EFFICIENCY SUMMARY  —  {PROC_EFF["label"]}')
print(f'OOD NLL on {ORACLE_FACTOR}× test set  (n = {N_SEEDS_EFF} seeds)')
print('='*70)

for model, col_short, col_oracle, col_t_short, col_t_oracle in [
    ('RoTHP', 'ro_short_ood', 'ro_oracle_ood', 't_ro_short', 't_ro_oracle'),
    ('HoTHP', 'ho_short_ood', 'ho_oracle_ood', 't_ho_short', 't_ho_oracle'),
]:
    s_nll  = edf[col_short].mean()
    o_nll  = edf[col_oracle].mean()
    gap    = s_nll - o_nll          # positive = short is worse; negative = short is BETTER
    ratio  = edf[col_t_oracle].mean() / edf[col_t_short].mean()
    saving = (1 - 1 / ratio) * 100

    print(f'\n  {model}')
    print(f'    Short-trained OOD NLL : {s_nll:.4f} ± {edf[col_short].std():.4f}')
    print(f'    Oracle-trained OOD NLL: {o_nll:.4f} ± {edf[col_oracle].std():.4f}')
    print(f'    NLL gap (short−oracle): {gap:+.4f} nats  '
          f'{"(short ≈ oracle)" if abs(gap) < 0.05 else ""}')
    print(f'    Oracle / short time   : {ratio:.1f}×  →  ~{saving:.0f}% training time saved')

print()
ho_gap  = (edf['ho_short_ood'] - edf['ho_oracle_ood']).mean()
ro_gap  = (edf['ro_short_ood'] - edf['ro_oracle_ood']).mean()
ho_ratio = edf['t_ho_oracle'].mean() / edf['t_ho_short'].mean()
print(f'  KEY CLAIM: HoTHP short-trained NLL gap vs oracle = {ho_gap:+.4f} nats')
print(f'  RoTHP short-trained NLL gap vs oracle            = {ro_gap:+.4f} nats')
print(f'  HoTHP oracle training is {ho_ratio:.1f}× more expensive than short training')
print(f'  → Training HoTHP on {TRAIN_LEN}-event sequences and deploying on '
      f'{ORACLE_LEN}-event sequences')
print(f'    costs ~{100/ho_ratio:.0f}% of the compute of full oracle training.')

# ── Figure ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: NLL comparison bar chart
ax = axes[0]
models   = ['RoTHP', 'HoTHP']
x_bar    = np.array([0, 1])
width    = 0.3
for i, (model, cs, co) in enumerate([
        ('RoTHP', 'ro_short_ood', 'ro_oracle_ood'),
        ('HoTHP', 'ho_short_ood', 'ho_oracle_ood')]):
    xb = x_bar[i]
    ax.bar(xb - width/2, edf[cs].mean(),  width, color='#78909C', alpha=0.9,
           label='Short-trained' if i == 0 else None,
           yerr=edf[cs].std(), capsize=5)
    ax.bar(xb + width/2, edf[co].mean(), width, color='#37474F', alpha=0.9,
           label='Oracle-trained' if i == 0 else None,
           yerr=edf[co].std(), capsize=5)
    gap = edf[cs].mean() - edf[co].mean()
    ax.text(xb, max(edf[cs].mean(), edf[co].mean()) + edf[cs].std() + 0.02,
            f'{gap:+.3f}', ha='center', fontsize=10,
            color='green' if gap < 0.05 else 'red')

ax.set_xticks(x_bar); ax.set_xticklabels(models, fontsize=12)
ax.set_ylabel(f'OOD NLL at {ORACLE_FACTOR}× horizon (nats)', fontsize=11)
ax.set_title('Short-trained ≈ Oracle?', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Right: time ratio
ax = axes[1]
ratios   = [
    edf['t_ro_oracle'].mean() / edf['t_ro_short'].mean(),
    edf['t_ho_oracle'].mean() / edf['t_ho_short'].mean(),
]
colors_bar = [C_RO, C_HO]
bars = ax.bar(models, ratios, color=colors_bar, alpha=0.85, width=0.4)
ax.axhline(1.0, color='black', ls='--', lw=1.2, alpha=0.5)
ax.set_ylabel(f'Oracle / Short training time  (×)', fontsize=11)
ax.set_title(f'Compute cost: oracle vs short training\n'
             f'(short = ≤{TRAIN_LEN} events,  oracle = ≤{ORACLE_LEN} events)',
             fontsize=12, fontweight='bold')
for bar, r in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width()/2, r + 0.1,
            f'{r:.1f}×', ha='center', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig.suptitle(
    '"Train Short, Test Long" — Efficiency Analysis\n'
    f'{PROC_EFF["label"]}  |  n = {N_SEEDS_EFF} seeds',
    fontsize=13, fontweight='bold', y=1.03,
)
plt.tight_layout()
plt.savefig('HoTHP_Efficiency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → HoTHP_Efficiency.png')